<a href="https://colab.research.google.com/github/Aeman-Imtiaz/Parallel-and-Distributed-Computing/blob/main/SemaphoreLec.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**SEMAPHORES**

A semaphore is a synchronization variable used to control access to a shared resource.


In [ ]:
%%bash
cat <<EOF > code.c
#include <stdio.h>
#include <pthread.h>
#include <semaphore.h>
#include <unistd.h>

sem_t sem;

void* worker(void* arg) {

    int id = *(int*)arg;

    printf("Thread %d wants to enter\n", id);

    // LOCK
    sem_wait(&sem);

    printf("Thread %d ENTERED critical section\n", id);

    sleep(2);

    printf("Thread %d LEAVING critical section\n", id);

    // UNLOCK
    sem_post(&sem);

    return NULL;
}

int main() {

    pthread_t t1, t2;
    int id1 = 1;
    int id2 = 2;

    // value = 1
    sem_init(&sem, 0, 1);

    pthread_create(&t1, NULL, worker, &id1);
    pthread_create(&t2, NULL, worker, &id2);

    pthread_join(t1, NULL);
    pthread_join(t2, NULL);

    sem_destroy(&sem);

    return 0;
}
EOF

gcc code.c -o code -pthread
./code

Thread 1 wants to enter
Thread 1 ENTERED critical section
Thread 2 wants to enter
Thread 1 LEAVING critical section
Thread 2 ENTERED critical section
Thread 2 LEAVING critical section


**Dry Run**

Initial semaphore value:

1
Thread 1
sem_wait(&sem);

Counter:

1 → 0

Thread 1 enters.

Thread 2

tries:

sem_wait(&sem);

Counter already 0.

So:

Thread 2 BLOCKS

Waits.

Thread 1 finishes
sem_post(&sem);

Counter:

0 → 1

Now Thread 2 wakes up.

Key Idea

Binary semaphore:

0 or 1

Like bathroom key:

key available → enter
no key → wait

In [ ]:
%%bash
cat <<EOF > code.c

#include <stdio.h>
#include <pthread.h>
#include <semaphore.h>
#include <unistd.h>

sem_t sem;  // Counting semaphore (allows 2 threads at once)

void* print_hello(void* arg) {
    int thread_id = *(int*)arg;

    printf("Thread %d: Trying to enter...\n", thread_id);
    sem_wait(&sem);  // Request permission (decrease counter)

    // CRITICAL SECTION - Only 2 threads can be here at a time
    printf(">>> Thread %d: INSIDE CRITICAL SECTION! <<<\n", thread_id);
    printf("Hello from thread %d\n", thread_id);
    sleep(2);  // Simulate work

    sem_post(&sem);  // Release permission (increase counter)
    printf("Thread %d: LEFT critical section\n", thread_id);

    return NULL;
}

int main() {
    pthread_t t1, t2, t3, t4;
    int id1 = 1, id2 = 2, id3 = 3, id4 = 4;

    // Initialize semaphore with value 2 (allows 2 threads at once)
    sem_init(&sem, 0, 2);

    // Create FOUR threads
    pthread_create(&t1, NULL, print_hello, &id1);
    pthread_create(&t2, NULL, print_hello, &id2);
    pthread_create(&t3, NULL, print_hello, &id3);
    pthread_create(&t4, NULL, print_hello, &id4);

    // Wait for all threads to finish
    pthread_join(t1, NULL);
    pthread_join(t2, NULL);
    pthread_join(t3, NULL);
    pthread_join(t4, NULL);

    // Clean up
    sem_destroy(&sem);

    return 0;
}

EOF

gcc code.c -lpthread -o code
./code

Thread 1: Trying to enter...
>>> Thread 1: INSIDE CRITICAL SECTION! <<<
Hello from thread 1
Thread 2: Trying to enter...
>>> Thread 2: INSIDE CRITICAL SECTION! <<<
Hello from thread 2
Thread 3: Trying to enter...
Thread 4: Trying to enter...
Thread 2: LEFT critical section
Thread 1: LEFT critical section
>>> Thread 4: INSIDE CRITICAL SECTION! <<<
Hello from thread 4
>>> Thread 3: INSIDE CRITICAL SECTION! <<<
Hello from thread 3
Thread 3: LEFT critical section
Thread 4: LEFT critical section


In [ ]:
%%bash
cat <<EOF > code.c
#include <stdio.h>
#include <pthread.h>
#include <semaphore.h>
#include <unistd.h>

sem_t sem;

void* task(void* arg) {

    int id = *(int*)arg;

    printf("Thread %d waiting...\n", id);

    sem_wait(&sem);

    printf("Thread %d ENTERED\n", id);

    sleep(3);

    printf("Thread %d EXITED\n", id);

    sem_post(&sem);

    return NULL;
}

int main() {

    pthread_t t[4];
    int ids[4] = {1,2,3,4};

    // allow 2 threads
    sem_init(&sem, 0, 2);

    for(int i=0; i<4; i++) {
        pthread_create(&t[i], NULL, task, &ids[i]);
    }

    for(int i=0; i<4; i++) {
        pthread_join(t[i], NULL);
    }

    sem_destroy(&sem);

    return 0;
}
EOF

gcc code.c -o code -pthread
./code

Thread 1 waiting...
Thread 1 ENTERED
Thread 2 waiting...
Thread 2 ENTERED
Thread 3 waiting...
Thread 4 waiting...
Thread 1 EXITED
Thread 3 ENTERED
Thread 2 EXITED
Thread 4 ENTERED
Thread 3 EXITED
Thread 4 EXITED


In [ ]:
%%bash
cat <<EOF > code.c
#include <stdio.h>
#include <pthread.h>
#include <semaphore.h>
#include <unistd.h>

sem_t full;
sem_t empty;

int item = 0;

void* producer(void* arg) {

    for(int i=1; i<=5; i++) {

        sem_wait(&empty);

        item = i;

        printf("Produced: %d\n", item);

        sem_post(&full);

        sleep(1);
    }

    return NULL;
}

void* consumer(void* arg) {

    for(int i=1; i<=5; i++) {

        sem_wait(&full);

        printf("Consumed: %d\n", item);

        sem_post(&empty);

        sleep(2);
    }

    return NULL;
}

int main() {

    pthread_t p, c;

    sem_init(&empty, 0, 1);
    sem_init(&full, 0, 0);
// sem_init(semaphore, shared, initial_value);>> initial value = 0 items available
    pthread_create(&p, NULL, producer, NULL);
    pthread_create(&c, NULL, consumer, NULL);

    pthread_join(p, NULL);
    pthread_join(c, NULL);

    sem_destroy(&empty);
    sem_destroy(&full);

    return 0;
}
EOF

gcc code.c -o code -pthread
./code

Produced: 1
Consumed: 1
Produced: 2
Consumed: 2
Produced: 3
Consumed: 3
Produced: 4
Consumed: 4
Produced: 5
Consumed: 5
